# Creating Baseline Models

We've cleaned our data, created some visuals to help gain some insights, and can now move on to creating some baseline models to predict the failure rate.

This notebook is mainly a first pass skim of baselines and is not exhaustive in finding the best possible model for our problem at the moment.

Starting off, we'll import our necessary libraries and data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline

The data we'll be using comes from the second notebook titled `02_feature_eng_preprocessing.ipynb`

In [ ]:
df = pd.read_csv('../data/processed/model_data.csv')
df.sample(5)

It would be a good idea to double check that there are no null or missing values

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe()

I had a problem before in another preprocessing step where I ended up with `inf` values in `failure_rate`, so the code commented out below took care of those values.

Looking at the data bove there doesn't seem to be any `inf` values anymore so we are all good on that front for now.

In [ ]:
# inf_values = df[df['failure_rate'] == np.inf]
# inf_values

In [ ]:
# df = df.drop(inf_values.index, axis=0)
# df.describe()

## Modeling Pipeline

Without messing around too much, I think it's time to jump in to fitting some basic models.

Here's how I'm going to structure the modeling process:
- split data into X and y datasets
- create train and test sets
- make a pipeline for each model with the following contained in it:
    - normalize the data with `StandardScaler()`
    - define a different model in each pipeline
        - we're predicting a numerical value so our problem is a regression type problem 
        - `LinearRegression()`
        - `RandomForestRegressor()`
        - `DecisionTreeRegressor()`
- fit our training data
- output the accuracy of our model

In [ ]:
X = df.drop('failure_rate', axis=1)
y = df['failure_rate']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

linear_pipeline = make_pipeline(StandardScaler(), LinearRegression())
linear_pipeline.fit(X_train, y_train)
linear_pipeline.score(X_test, y_test)

In [ ]:
forest_pipeline = make_pipeline(StandardScaler(), RandomForestRegressor())
forest_pipeline.fit(X_train, y_train)
forest_pipeline.score(X_test, y_test)

In [ ]:
dectree_pipeline = make_pipeline(StandardScaler(), DecisionTreeRegressor())
dectree_pipeline.fit(X_train, y_train)
dectree_pipeline.score(X_test, y_test)

We can see that we get the best results with our tree-based models, and if we keep re-running these two tree models then we'll get different scores each time. With the scores varying, let's pluck out which features each model thinks is the most important for determining failure rate.

## Feature Importances

To plot the feature importances, we'll create an importance variable and then sort them from highest to lowest values. Using matplotlib, we create a bar plot using the range of features from the training dataset, but only for the 15 numerical values in the data, and input the sorted importances.

In [ ]:
importances = forest_pipeline.steps[1][1].feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 5))
plt.title("Feature importances")
plt.bar(range(X_train.shape[1]), importances[indices], color="r", align="center")
plt.xticks(range(X_train.shape[1]), X_train.columns[indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.show();

In [ ]:
importances = dectree_pipeline.steps[1][1].feature_importances_ 
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 5))
plt.title("Feature importances")
plt.bar(range(X_train.shape[1]), importances[indices], color="r", align="center")
plt.xticks(range(X_train.shape[1]), X_train.columns[indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.show();

In these next steps I'm going to load in a dataset that still has the latitude and longitude data, I'll fit it into a random forest model and then visualize the predictions that we get from it.

In [ ]:
final_data = pd.read_csv('../data/interim/final_data.csv')
final_data.sample(5)

In [ ]:
final_data[final_data['failure_rate'] == np.inf]

In [ ]:
inf_values = final_data[final_data['failure_rate'] == np.inf]
final_data = final_data.drop(inf_values.index, axis=0)

In [ ]:
final_data.drop(['incident_date', 'asset_year_installed'], axis=1, inplace=True)

In [ ]:
final_X = final_data.drop('failure_rate', axis=1)
final_y = final_data['failure_rate']

X_train, X_test, y_train, y_test = train_test_split(final_X, final_y, test_size=0.2, random_state=42)
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
rf.score(X_test, y_test)

In [ ]:
y_pred = rf.predict(X_test)

# save the predictions to a csv file
predictions = pd.DataFrame(y_pred, columns=['predictions'])
predictions.to_csv('../data/processed/predictions.csv', index=False)

# add the predictions to the test data and save it as a new csv file
X_test['predictions'] = y_pred
X_test.to_csv('../data/processed/test_predict_data.csv', index=False)

plt.figure(figsize=(10, 5))
plt.scatter(X_test['longitude'], X_test['latitude'], c=y_pred, cmap='viridis')
plt.colorbar()
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show();

In [ ]:
import config
import plotly.express as px
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

In [ ]:
px.set_mapbox_access_token(config.token)
token = config.token
fig = px.scatter_mapbox(data_frame=X_test, lat='latitude', lon='longitude',
                        color='age_at_break', size=y_pred.round(2), hover_name=y_pred.round(2),
                        color_continuous_scale=px.colors.cyclical.IceFire)
fig.update_layout(mapbox_style='carto-positron', mapbox_accesstoken=token)
fig.show(renderer='notebook_connected')

In [ ]:
import gmaps
import gmaps.datasets
gmaps.configure(api_key=config.gmaps_api_key)

In [ ]:
lat = X_test['latitude'].values
lon = X_test['longitude'].values
break_locations = list(zip(lat, lon))
fig = gmaps.figure()
# fig.add_layer(gmaps.heatmap_layer(break_locations, weights=y_pred))
break_map = gmaps.heatmap_layer(break_locations, weights=y_pred)
fig.add_layer(break_map)
fig

In [ ]:
from IPython.display import Image
Image(filename='../figures/predictions_map.png')